In [ ]:
import polars as pl

In [ ]:
WELLHUB_PATH = "/Users/emulie/Data/Wellhub/mixpanel_create_record_wellhub_20260511.json"
df = pl.read_json(WELLHUB_PATH)

In [ ]:
# df

In [ ]:
series = df.select('series').item()

In [ ]:
# series

In [ ]:
rows = []
for user_id, user_data in series['Uniques of create_record_wellhub'].items():
    for country, country_data in user_data.items():
        try: 
            if country == "$overall":
                continue
            for date, metrics in country_data.items():
                if date == "$overall":
                    continue
                rows.append({
                    "date": date, 
                    "country": country,
                    "user_id": user_id, 
                    "count": metrics.get("all", 0),
                })
        except:
            pass
results = pl.DataFrame(rows)

In [ ]:
results = results.with_columns(
    pl.col("date").str.to_date().alias("date")
).with_columns(
    pl.col("date").dt.year().alias("year"),
    pl.col("date").dt.month().alias("month"),
    pl.col("count").clip(upper_bound=1) # check if users has DUA
)

In [ ]:
results

In [ ]:
dua = results.group_by(["year", "month", "user_id", "country"]).agg(
    pl.col("count").sum().alias("DUA")
)

In [ ]:
dua

In [ ]:
# get MAU per country
dua.group_by(["year", "month", "country"]).agg(
    pl.col("user_id").count().alias("MAU")
)

In [ ]:
print(dua['country'].unique())

In [ ]:
# get revenue
# Rates by region:
# 🇺🇸 US, 🇬🇧 UK, 🇮🇪 IE, 🇩🇪 DE, 🇪🇸 ES, 🇮🇹 IT → $1.75 per DUA (capped at $8.75/user/month = 5 DUAs max)
# 🇧🇷 Brazil → $1.50 per DUA (capped at $7.50/user/month)
# 🇲🇽 MX, 🇨🇱 CL, 🇦🇷 AR → $1.25 per DUA (capped at $6.25/user/month)

COUNTRIES_TIER_175 = ["United States", "United Kingdom", "Ireland", "Germany", "Spain", "Italy"]
COUNTRIES_TIER_150 = ["Brazil"]
COUNTRIES_TIER_125 = ["Mexico", "Chile", "Argentina"]

revenues = (
    dua
    # cap monthly DUA to 5
    .with_columns(
        pl.col("DUA").clip(upper_bound=5).alias("capped_dua")
    )
    # apply country pricing
    .with_columns(
        pl.when(pl.col("country").is_in(COUNTRIES_TIER_175))
                .then(pl.col("capped_dua") * 1.75)
          .when(pl.col("country").is_in(COUNTRIES_TIER_150))
                .then(pl.col("capped_dua") * 1.50)
          .when(pl.col("country").is_in(COUNTRIES_TIER_125))
                .then(pl.col("capped_dua") * 1.25)
          .otherwise(1.25) # TODO: ask what to do for users in countries not included
          .alias("revenue")
    )
)

In [ ]:
revenues.group_by(["year", "month"]).agg(
    pl.col("revenue").sum().alias("revenue")
)

### Using create_record_wellhub only

In [ ]:
# WELLHUB_OVERALL_PATH = "/Users/emulie/Downloads/Daily wellhub usage_Insights_2026-04-11_to_2026-05-11.csv" # all create_record_wellhub w/ groups = wellhub
# WELLHUB_OVERALL_PATH = "/Users/emulie/Downloads/Daily wellhub usage_Insights_2026-04-11_to_2026-05-11 (1).csv" # all create_record_wellhub 
WELLHUB_OVERALL_PATH = "/Users/emulie/Downloads/Daily wellhub usage_Insights_2026-04-01_to_2026-04-30.csv" # all create_record_wellhub w/ groups = wellhub from April 1st to 310st

df = pl.read_csv(WELLHUB_OVERALL_PATH)

In [ ]:
df = (
    df
    .rename({"User ID": "user_id", "Country": "country", "Date YYYY-MM-DD": "date", "Uniques of create_record_wellhub": "create_record_wellhub"})
    .with_columns(
        pl.col("date").str.to_date().alias("date"), 
    )
    .with_columns(
        pl.col("date").dt.year().alias("year"),
        pl.col("date").dt.month().alias("month"),
        pl.col("create_record_wellhub").clip(upper_bound=1).alias("count")
    )
)

In [ ]:
df

In [ ]:
results = df

In [ ]:
dua = results.group_by(["year", "month", "user_id", "country"]).agg(
    pl.col("count").sum().alias("DUA")
)

In [ ]:
mau = dua.group_by(["year", "month", "country"]).agg(
    pl.col("user_id").count().alias("MAU")
)

In [ ]:
# get revenue
# Rates by region:
# 🇺🇸 US, 🇬🇧 UK, 🇮🇪 IE, 🇩🇪 DE, 🇪🇸 ES, 🇮🇹 IT → $1.75 per DUA (capped at $8.75/user/month = 5 DUAs max)
# 🇧🇷 Brazil → $1.50 per DUA (capped at $7.50/user/month)
# 🇲🇽 MX, 🇨🇱 CL, 🇦🇷 AR → $1.25 per DUA (capped at $6.25/user/month)

COUNTRIES_TIER_175 = ["United States", "United Kingdom", "Ireland", "Germany", "Spain", "Italy"]
COUNTRIES_TIER_150 = ["Brazil"]
COUNTRIES_TIER_125 = ["Mexico", "Chile", "Argentina"]

revenues = (
    dua
    # cap monthly DUA to 5
    .with_columns(
        pl.col("DUA").clip(upper_bound=5).alias("capped_dua")
    )
    # apply country pricing
    .with_columns(
        pl.when(pl.col("country").is_in(COUNTRIES_TIER_175))
                .then(pl.col("capped_dua") * 1.75)
          .when(pl.col("country").is_in(COUNTRIES_TIER_150))
                .then(pl.col("capped_dua") * 1.50)
          .when(pl.col("country").is_in(COUNTRIES_TIER_125))
                .then(pl.col("capped_dua") * 1.25)
          .otherwise(1.25) # TODO: ask what to do for users in countries not included
          .alias("revenue")
    )
)

In [ ]:
revenues.group_by(["year", "month"]).agg(
    pl.col("revenue").sum().alias("revenue")
)

In [ ]:
len(set(df['country'].unique()) - set(COUNTRIES_TIER_175 + COUNTRIES_TIER_150 + COUNTRIES_TIER_125))

In [ ]:
missing_countries = set(df['country'].unique()) - set(COUNTRIES_TIER_175 + COUNTRIES_TIER_150 + COUNTRIES_TIER_125)

In [ ]:
len(set(df['country'].unique()))

### EDA

- How many DUAs in April came from countries outside those three tiers?

In [ ]:
mau.filter(pl.col("country").is_in(missing_countries))['MAU'].sum()

In [ ]:
dua.filter(pl.col("country").is_in(missing_countries))['DUA'].sum()

In [ ]:
# save
df.write_csv("create_record_wellhub_by_user_20260401_20260430.csv")
mau.write_csv("monthly_wellhub_users_by_country_April2026.csv")
dua.write_csv("monthly_user_action_by_user_id_April2026.csv")